# `sentiment_gate3.py` — Playground

Manual verification notebook for **Gate 3: Sentiment Quality Check** (Claude call 2 of 4).

| Function | Status | Notes |
|---|---|---|
| `evaluate_gate3_sentiment(candidate, headlines)` | ✅ built | One focused Claude question: is the news sentiment bullish, and how confident? |

**Output:** `{passed, direction, confidence, key_reason, caution, size_reduction_pct}`

Claude returns `direction` (BULLISH/BEARISH/NEUTRAL) + `confidence` (0–10) + `key_reason`. **Confidence is the single quality signal** — each headline is tagged `[Reliability: …]` in the prompt, so Claude folds source quality into its confidence rather than reporting reliability separately. The pass/block/caution verdict is made in code by `helpers/logic/sentiment_rules.apply_pass_rules`.

**Decision logic (`apply_pass_rules`):**

| Situation | Result |
|---|---|
| BEARISH | `passed=False` (BLOCK) |
| NEUTRAL + confidence < 6 | `passed=False` (BLOCK) |
| NEUTRAL + confidence ≥ 6 | `passed=True`, `caution=True`, −25% size |
| BULLISH + confidence < 6 | `passed=True`, `caution=True`, −25% size |
| BULLISH + confidence ≥ 6 | `passed=True` (clean) |
| Empty headlines | `passed=False`, `key_reason='no_headlines'`, no LLM call (no news = nothing to confirm) |
| LLM unavailable | `passed=False`, `key_reason='llm_unavailable'` (don't trade blind) |

**Standard used:** `helpers/llm/client.build_agent` + `run_agent` (cheap Haiku default, per-gate overridable). Live calls need `ANTHROPIC_API_KEY` in `.env`.

In [1]:
import sys
import pathlib

# Add 02_intelligence/ (for helpers) and gate3_sentiment/ (for sentiment_gate3.py directly).
intelligence_dir = pathlib.Path('backend/02_intelligence').resolve()
gate3_dir        = intelligence_dir / 'gate3_sentiment'

for p in [str(intelligence_dir), str(gate3_dir)]:
    if p not in sys.path:
        sys.path.insert(0, p)

from sentiment_gate3 import evaluate_gate3_sentiment
from helpers.fetchers.news import fetch_news

---
## Happy path — bullish headlines → PASS

Strong positive news from HIGH/MEDIUM sources. Expect `direction=BULLISH`, high confidence, `passed=True`, no caution.

In [ ]:
candidate = {'ticker': 'NVDA'}

bullish = [
    {'headline': 'NVIDIA beats earnings, raises guidance on record data-center demand',
     'source': 'Reuters', 'reliability': 'HIGH', 'summary': ''},
    {'headline': 'Analysts hike NVIDIA price targets after blowout quarter',
     'source': 'CNBC', 'reliability': 'MEDIUM', 'summary': ''},
]

evaluate_gate3_sentiment(candidate, headlines=bullish)

---
## Variation — bearish headlines → BLOCK

Negative outlook + downgrades from HIGH sources. Expect `direction=BEARISH`, `passed=False`.

In [3]:
bearish = [
    {'headline': 'NVIDIA shares slide as demand outlook darkens and orders are cut',
     'source': 'Bloomberg', 'reliability': 'HIGH', 'summary': ''},
    {'headline': 'Analysts downgrade NVIDIA, warn of margin pressure ahead',
     'source': 'Reuters', 'reliability': 'HIGH', 'summary': ''},
]

evaluate_gate3_sentiment(candidate, headlines=bearish)

[gate3] NVDA: BLOCKED — BEARISH conf=9: Both high-reliability sources report deteriorating demand outlook, order cuts, and analyst downgrades with margin pressure concerns, creating a strongly negative sentiment for NVIDIA.


{'passed': False,
 'direction': 'BEARISH',
 'confidence': 9,
 'key_reason': 'Both high-reliability sources report deteriorating demand outlook, order cuts, and analyst downgrades with margin pressure concerns, creating a strongly negative sentiment for NVIDIA.',
 'caution': False,
 'size_reduction_pct': 0}

---
## Variation — weak / low-reliability headlines → low confidence

Vague, unsourced chatter from LOW-tier sources. Claude should report **low confidence** (it folds the weak source tags into the number) — which yields a BLOCK (NEUTRAL conf < 6) or a caution pass (BULLISH conf < 6). Watch confidence drive the verdict.

In [4]:
mixed = [
    {'headline': 'NVIDIA rumored to be working on a new consumer product, blogger says',
     'source': 'Some Blog', 'reliability': 'LOW', 'summary': ''},
    {'headline': 'Retail traders bullish on NVIDIA in forum chatter',
     'source': 'stocktwits', 'reliability': 'LOW', 'summary': ''},
]

evaluate_gate3_sentiment(candidate, headlines=mixed)

[gate3] NVDA: BLOCKED — NEUTRAL conf=2: Both headlines come from low-reliability sources (blog rumors and retail forum chatter) with no substantive news or catalysts to support a bullish momentum entry.


{'passed': False,
 'direction': 'NEUTRAL',
 'confidence': 2,
 'key_reason': 'Both headlines come from low-reliability sources (blog rumors and retail forum chatter) with no substantive news or catalysts to support a bullish momentum entry.',
 'caution': False,
 'size_reduction_pct': 0}

---
## Real news — multiple tickers

The genuine end-to-end path (live `fetch_news` + live LLM), not synthetic fixtures. Three tickers across different sectors — Tesla, Apple, Pfizer — so you can see real variation in direction, confidence, and the resulting verdict. Results change with the news window.

In [ ]:
# Real news across multiple tickers — the pipeline shape: the caller fetches once per ticker,
# then passes the headlines into the gate. Live API + live LLM, so verdicts vary with the news window.
for tkr in ['TSLA', 'AAPL', 'PFE']:
    headlines = fetch_news(tkr) or []
    r = evaluate_gate3_sentiment({'ticker': tkr}, headlines)
    verdict = 'PASS' if r['passed'] else 'BLOCK'
    caution = ' (caution -%d%%)' % r['size_reduction_pct'] if r['caution'] else ''
    print(f"{tkr:<5} {verdict}{caution:<16} dir={r['direction']:<8} conf={r['confidence']}")

In [ ]:
for tkr in ['TSLA']:
    headlines = fetch_news(tkr) or []
    r = evaluate_gate3_sentiment({'ticker': tkr}, headlines)
    verdict = 'PASS' if r['passed'] else 'BLOCK'
    caution = ' (caution -%d%%)' % r['size_reduction_pct'] if r['caution'] else ''
    print(f"{tkr:<5} {verdict}{caution:<16} dir={r['direction']:<8} conf={r['confidence']}")

In [ ]:
headlines = fetch_news('PFE') or []

r = evaluate_gate3_sentiment({'ticker': 'PFE'}, headlines)
r

---
## Failure / edge — empty headlines → BLOCK, no LLM call

No news means no bullish sentiment to confirm, so Gate 3 blocks immediately without spending a Claude call (`key_reason='no_headlines'`).

In [6]:
evaluate_gate3_sentiment(candidate, headlines=[])

[gate3] NVDA: no headlines — blocking (cannot confirm sentiment)


{'passed': False,
 'direction': '',
 'confidence': 0,
 'key_reason': 'no_headlines',
 'caution': False,
 'size_reduction_pct': 0}

---
## Free-play

Try your own headlines, tickers, or edge cases below.